In [1]:
ConnectString = "mysql -h database-1.clqxqhhe6wft.us-east-1.rds.amazonaws.com -P 3306 -u admin -p'<Enter_DB_Password>' --ssl-verify-server-cert  --ssl-ca=/certs/global-bundle.pem mysql"

host=ConnectString[9:60]
user='admin'
password='Data608-Project'
database='' # Optional: leave out to create DB first


In [2]:
!pip install mysql-connector-python --break-system-packages

Defaulting to user installation because normal site-packages is not writeable


In [3]:
import mysql.connector
from mysql.connector import Error

In [4]:
connection = mysql.connector.connect(
    host=host,
    user=user,
    password=password,
    database='' # Optional: leave out to create DB first
   )

if connection.is_connected():
    db_info = info = connection.server_info
    print(f"Connected to MySQL Server version: {db_info}")
    cursor = connection.cursor()


Connected to MySQL Server version: 8.4.7


In [5]:

# 2. Create a cursor
cursor = connection.cursor()

# 3. Execute the CREATE DATABASE command
# Use "IF NOT EXISTS" to prevent errors if the DB already exists
cursor.execute("CREATE DATABASE IF NOT EXISTS CHESSBOT")

print("Database created successfully!")

# 4. (Optional) Switch to the new database to start using it
#cursor.execute("USE my_new_database")
# Alternatively, re-assign the database property directly:
# mydb.database = 'my_new_database'


Database created successfully!


In [6]:
# 1. Execute the command
cursor.execute("SHOW DATABASES")

# 2. Fetch and print the results
print("Available Databases:")
for (db_name,) in cursor:
    print(f"- {db_name}")

Available Databases:
- CHESSBOT
- information_schema
- mysql
- performance_schema
- sys


In [7]:
#select database to use
cursor = connection.cursor()
cursor.execute("USE CHESSBOT")
print("Selected CHESSBOT database")

Selected CHESSBOT database


In [8]:
#cursor.execute("DROP TABLE IF EXISTS session")
#cursor.execute("DROP TABLE IF EXISTS moves")
#cursor.execute("DROP TABLE IF EXISTS games")

In [9]:
# --- CREATE: Create a table ---
games_table_name = 'games'
moves_table_name = 'moves'
session_table_name = 'session'

gamestableexists = False
movestableexists = False
sessiontableexists = False

# 1. Execute the command to check if table exists
cursor.execute("SHOW TABLES")

# 2. Fetch and print the results, capture table exists
print("Available Tables:")
for (db_name,) in cursor:
    print(f"- {db_name}")
    if(db_name == games_table_name):
        gamestableexists = True
        print ("-------------")
        print ("table: ",games_table_name," already exists so will not recreate")
    if(db_name == moves_table_name):
        movestableexists = True
        print ("-------------")
        print ("table: ",moves_table_name," already exists so will not recreate")
    if(db_name == session_table_name):
        sessiontableexists = True
        print ("-------------")
        print ("table: ",session_table_name," already exists so will not recreate")


if gamestableexists == False:   # create games table
    print ("-------------")
    print ("table: ",games_table_name," does not exists")
    CreateTable = "CREATE TABLE " + games_table_name + "(id INT AUTO_INCREMENT PRIMARY KEY,Site VARCHAR(50),White_ELO INT,Black_ELO INT,Opening VARCHAR(100))"
    print(CreateTable)
    cursor.execute(CreateTable)
    print("Table ",games_table_name, " created.")    

if movestableexists == False:   # create moves table
    print ("-------------")
    print ("table: ",moves_table_name," does not exists")
    CreateTable2 = "CREATE TABLE " + moves_table_name + "(id INT AUTO_INCREMENT PRIMARY KEY,Set5 VARCHAR(75),Set10 VARCHAR(75),Set15 VARCHAR(75),Set20 VARCHAR(75),Set30 VARCHAR(150),Set40 VARCHAR(150),Set70 VARCHAR(450),Set_Remain VARCHAR(450),BlackELO INT,Game_id INT,FOREIGN KEY (Game_id) REFERENCES games(id))"
    print(CreateTable2)
    cursor.execute(CreateTable2)
    print("Table ",moves_table_name, " created.")    

    ### create Move indexes
    cursor.execute("CREATE INDEX idx_set5 ON moves(Set5)")
    cursor.execute("CREATE INDEX idx_set10 ON moves(Set10)")
    cursor.execute("CREATE INDEX idx_set15 ON moves(Set15)")
    cursor.execute("CREATE INDEX idx_set20 ON moves(Set20)")
    cursor.execute("CREATE INDEX idx_set30 ON moves(Set30)")
    print ("-------------")
    print ("Moves table indexes are created")

if sessiontableexists == False:   # create session table
    print ("-------------")
    print ("table: ",session_table_name," does not exists")
    CreateTable3 = "CREATE TABLE " + session_table_name + "(id INT AUTO_INCREMENT PRIMARY KEY,Cookie VARCHAR(255),Date_Time DATETIME,Game_id INT,Param_colour VARCHAR(50),Param_difficult VARCHAR(50),Param_metric VARCHAR(50),Param_snark VARCHAR(50),progress INT,result VARCHAR(255),FOREIGN KEY (Game_id) REFERENCES games(id))"
    print(CreateTable3)
    cursor.execute(CreateTable3)
    print("Table ",session_table_name, " created.")    


Available Tables:
- games
-------------
table:  games  already exists so will not recreate
- moves
-------------
table:  moves  already exists so will not recreate
- session
-------------
table:  session  already exists so will not recreate


In [10]:
#check games table

cursor.execute("DESCRIBE "+games_table_name)

# Fetch and print the headers
print(f"{'Field':<20} {'Type':<15} {'Null':<10} {'Key':<5}")
print("-" * 50)

for row in cursor.fetchall():
    print(f"{row[0]:<20} {row[1]:<15} {row[2]:<10} {row[3]:<5}")

Field                Type            Null       Key  
--------------------------------------------------
id                   int             NO         PRI  
Site                 varchar(50)     YES             
White_ELO            int             YES             
Black_ELO            int             YES             
Opening              varchar(100)    YES             


In [11]:
#check moves table

cursor.execute("DESCRIBE "+moves_table_name)

# Fetch and print the headers
print(f"{'Field':<20} {'Type':<15} {'Null':<10} {'Key':<5}")
print("-" * 50)

for row in cursor.fetchall():
    print(f"{row[0]:<20} {row[1]:<15} {row[2]:<10} {row[3]:<5}")

Field                Type            Null       Key  
--------------------------------------------------
id                   int             NO         PRI  
Set5                 varchar(75)     YES        MUL  
Set10                varchar(75)     YES        MUL  
Set15                varchar(75)     YES        MUL  
Set20                varchar(75)     YES        MUL  
Set30                varchar(150)    YES        MUL  
Set40                varchar(150)    YES             
Set70                varchar(450)    YES             
Set_Remain           varchar(450)    YES             
BlackELO             int             YES             
Game_id              int             YES        MUL  


In [12]:
#check session table

cursor.execute("DESCRIBE "+session_table_name)

# Fetch and print the headers
print(f"{'Field':<20} {'Type':<15} {'Null':<10} {'Key':<5}")
print("-" * 50)

for row in cursor.fetchall():
    print(f"{row[0]:<20} {row[1]:<15} {row[2]:<10} {row[3]:<5}")

Field                Type            Null       Key  
--------------------------------------------------
id                   int             NO         PRI  
Cookie               varchar(255)    YES             
Date_Time            datetime        YES             
Game_id              int             YES        MUL  
Param_colour         varchar(50)     YES             
Param_difficult      varchar(50)     YES             
Param_metric         varchar(50)     YES             
Param_snark          varchar(50)     YES             
progress             int             YES             
result               varchar(255)    YES             


In [13]:
# --- READ: Fetch the record count of games table ---
cursor.execute(f"SELECT COUNT(*) FROM {games_table_name}")
count = cursor.fetchone()[0]
print(f"{games_table_name} table has entries: {count}")


games table has entries: 27126


In [14]:
# --- READ: Fetch the record count of moves table ---
cursor.execute(f"SELECT COUNT(*) FROM {moves_table_name}")
count = cursor.fetchone()[0]
print(f"{moves_table_name} table has entries: {count}")


moves table has entries: 27126


In [15]:
# --- READ: Fetch the record count of session table ---
cursor.execute(f"SELECT COUNT(*) FROM {session_table_name}")
count = cursor.fetchone()[0]
print(f"{session_table_name} table has entries: {count}")


session table has entries: 0


In [16]:
#Michael's sandbox..
#INSERT

insert_query = "INSERT INTO " + games_table_name + "(Site, White_ELO,Black_ELO,Opening) VALUES (%s, %s, %s, %s)"
cursor.execute(insert_query, ("https://lichess.org/gx3qb4ur", 1155,1096,"King's Indian Attack"))
connection.commit()  # Required to save changes
InsertedRecord = cursor.lastrowid
print(f"Record inserted. ID: {InsertedRecord}")


Record inserted. ID: 27128


In [20]:
#Michael's sandbox..
#READ

cursor.execute("SELECT * FROM " + moves_table_name + " limit 0,13")
records = cursor.fetchall()
print(f"Total rows: {cursor.rowcount}")
for row in records:
    print("---------------")
    for value in row:
        print(str(value))



Total rows: 13
---------------
1
1. e2-e4 e7-e6 2. d2-d4 b7-b6 3. a2-a3 Bc8-b7 4. Nb1-c3 Ng8-h6 5. Bc1xh6 g7
6. Bf1-e2 Qd8-g5 7. Be2-g4 h6-h5 8. Ng1-f3 Qg5-g6 9. Nf3-h4 Qg6-g5 10. Bg4x
11. Qd1-f3 Ke8-d8 12. Qf3xf7 Nb8-c6 13. Qf7-e8#
None
None
None
None
None
1403
2
---------------
2
1. e2-e4 g7-g6 2. d2-d4 d7-d6 3. Ng1-f3 c7-c6 4. h2-h3 Ng8-f6 5. Bc1-g5 Nf6
6. Qd1-e2 Bc8-f5 7. Nb1-d2 Qd8-a5 8. c2-c3 Ne4xd2 9. Bg5xd2 Nb8-d7 10. b2-b
11. Nf3-g5 h7-h5 12. Qe2-c4 d6-d5 13. Qc4-e2 Qa3-b2 14. Qe2-d1 Bf5-c2 15. Q
16. Ra1xc1 Bc2-a4 17. Bf1-d3 Nd7-b6 18. Ke1-g1 Nb6-c4 19. Bd3xc4 d5xc4 20. 
21. Rf1-e1 Ke8-g8 22. Re1xe7 Ra8-e8 23. Re7xb7 f7-f6 24. Ng5-e6 Re8xe6 25. Bf4xh6 Rf8-f7 26. Rb7-b8+ Kg8-h7 27. Bh6-f4 g6-g5 28. Bf4-d2 Re6-e2 29. Bd2
31. Rb8-c8 Bc2-d3 32. Rc8xc6 Re2-c2+ 33. Kf1-g1 Rc2xc1 34. Rc6xf6 h5-h4 35. g2-g4 Re7xe1+ 36. Kg1-g2 Bd3-e4+ 37. f2-f3 Rc1-c2#
None
None
1544
3
---------------
3
1. b2-b3 e7-e5 2. Bc1-b2 e5-e4 3. d2-d3 Ng8-f6 4. Ng1-h3 d7-d5 5. d3xe4 Nf6
6. Nh3-f4 Qd8-h4 7. g2-g

In [18]:
#Michael's sandbox..
#UPDATE

update_query = "UPDATE games SET Opening = %s WHERE id = %s"
cursor.execute(update_query, ("Some new death method", InsertedRecord))
connection.commit()
print("Record updated.")


Record updated.


In [19]:
#Michael's sandbox..
#DELETE

delete_query = "DELETE FROM games WHERE id = %s"
cursor.execute(delete_query, (InsertedRecord,))
connection.commit()
print("Record deleted.")


Record deleted.
